# Benchmark Notebook — MiniBench (Final)
Gemini generates candidate functions; notebook scores them with deterministic unit tests.

Locked parameters match `golden_solution_filled.ipynb`.

In [ ]:
# =========================================================
# Candidate generation + load (Gemini -> exec -> functions)
# =========================================================
# Keep notebook runnable even if Gemini API is unavailable.
# Set to TRUE to run live Gemini agent.

USE_GEMINI_AGENT = False

# =========================================================
# Code Start
# =========================================================

import os, re, math
import numpy as np

SIGMA = 5.670374419e-8

ALPHA_WARM = 0.30
ALPHA_COLD = 0.62
T0_TRUE = 273.15
K_TRUE  = 5.0

S0 = 1361.0

# Subtask A
S_A = S0
BRACKET_A = (210.0, 330.0)
TOL_A = 1e-8
MAX_ITER_A = 200

# Subtask B
C = 2.5e8
A_SEASON = 0.10
P_DAYS = 365.0
DT_DAYS = 0.25
N_DAYS = int(365.0 * 8.0)
T_INIT = 288.0

# Subtask C
OBS_SEED = 42
NOISE_STD = 0.3
OBS_EVERY_DAYS = 5.0
T0_BOUNDS = (265.0, 285.0)
K_BOUNDS  = (2.0, 10.0)
times_days = np.arange(365.0*6.0, 365.0*8.0, OBS_EVERY_DAYS)


In [ ]:

BENCHMARK_PROMPT = """
You are solving a scientific computing benchmark. Implement the following three Python functions exactly as specified.

General rules:
- Use Python 3.
- You may use numpy and math. Do NOT rely on network or file I/O.
- If you use randomness, you MUST set a seed so results are deterministic.
- All outputs must be deterministic for fixed inputs.
- Implement functions with the exact signatures below.

Subtask A — Equilibrium temperature (root finding)
Implement:
def equilibrium_temperature(S, sigma, alpha_warm, alpha_cold, T0, k, bracket=(150.0, 350.0), tol=1e-8, max_iter=200):
    # Return T_star (float) that solves:
    #   (1 - alpha(T)) * S/4 - sigma*T^4 = 0
    # where
    #   alpha(T) = alpha_warm + (alpha_cold - alpha_warm)/(1 + exp((T - T0)/k))
    # Requirements:
    #   - Use a bracketed root-finding method (bisection or Brent).
    #   - If the bracket does not contain a root (no sign change), raise ValueError.
    #   - Enforce physical bounds: T>0 and 0<=alpha(T)<=1 (clamp alpha if needed).

Subtask B — Seasonal ODE simulation
Implement:
def simulate_temperature(C, S0, a, P_days, sigma, alpha_warm, alpha_cold, T0, k, T_init=288.0, dt_days=0.25, n_days=365*6):
    # Simulate:
    #   C dT/dt = (1-alpha(T))*S(t)/4 - sigma*T^4
    #   S(t)=S0*(1 + a*cos(2*pi*t/P))
    # Return dict with:
    #   - T_mean: mean temperature over the final 365 days
    #   - T_amp: (max - min)/2 over the final 365 days
    # Notes:
    #   - Use a stable integrator (Euler with small dt or RK4).
    #   - Ensure T stays > 0 and alpha(T) stays in [0,1].
    #   - Your output must be deterministic.

Subtask C — Calibrate albedo transition parameters
Implement:
def calibrate_T0_k(observations, times_days, C, S0, a, P_days, sigma, alpha_warm, alpha_cold, T_init=288.0, dt_days=0.25, search_bounds=((250.0, 290.0), (2.0, 20.0))):
    # Given observed temperatures at times_days, estimate (T0, k) by minimizing SSE:
    #   SSE = sum((T_model(t_i) - obs_i)^2)
    # Return dict with keys: T0_hat, k_hat, sse
    # Requirements:
    #   - Deterministic: same inputs => same outputs.
    #   - Prefer a deterministic method (e.g., 2-stage grid search).
    #   - Must beat a naive baseline guess (midpoint of bounds) in SSE.

Return ONLY a single Python code block containing the three function definitions and any helper functions.
"""


In [ ]:
# ============================================================
# SUBTASK A
# ============================================================

import os, re, time
import google.generativeai as genai

def extract_python_code_block(text: str) -> str:
    if not text:
        return ""
    blocks = re.findall(r"```python\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if blocks:
        return "\n\n".join(b.strip() for b in blocks if b.strip())
    blocks = re.findall(r"```\s*(.*?)```", text, flags=re.DOTALL)
    if blocks:
        return "\n\n".join(b.strip() for b in blocks if b.strip())
    return text.strip()

def call_gemini_for_code(prompt: str, *, model_name: str = None) -> str:
    api_key = os.environ.get("GEMINI_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("Missing GEMINI_API_KEY in environment.")
    model_name = model_name or os.environ.get("GEMINI_MODEL", "gemini-1.5-flash")
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel(model_name)
    resp = model.generate_content(
        prompt,
        generation_config={"temperature": 0.2, "max_output_tokens": 2500},
    )
    return getattr(resp, "text", "") or ""

def load_candidate_code() -> str:
    if USE_GEMINI_AGENT:
        # Keep the prompt short to reduce latency.
        code_only = (
            "Return ONLY Python code (no markdown/backticks) defining exactly three functions:\n"
            "equilibrium_temperature(...), simulate_temperature(...), calibrate_T0_k(...).\n"
            "Use numpy+math only. Deterministic only. No prints.\n\n"
            + BENCHMARK_PROMPT
        )
        try:
            t0 = time.time()
            txt = call_gemini_for_code(code_only)
            print("Gemini returned chars:", len(txt), "in", round(time.time() - t0, 2), "s")
            return extract_python_code_block(txt)
        except Exception as e:
            print("Gemini unavailable; falling back to candidate_code.py")
            print("Reason:", repr(e))

    with open("candidate_code.py", "r", encoding="utf-8") as f:
        return f.read()

candidate_code = load_candidate_code()
if not candidate_code.strip():
    raise RuntimeError("candidate_code is empty (Gemini + fallback both failed).")

try:
    exec(compile(candidate_code, "<candidate_code>", "exec"), globals(), globals())
except Exception as e:
    print("=== Candidate code failed to exec ===")
    print("Error:", repr(e))
    print("\n--- Candidate code (first 2000 chars) ---\n")
    print(candidate_code[:2000])
    raise

required = ["equilibrium_temperature", "simulate_temperature", "calibrate_T0_k"]
missing = [fn for fn in required if fn not in globals() or not callable(globals().get(fn))]
if missing:
    print("\n--- Candidate code (first 2000 chars) ---\n")
    print(candidate_code[:2000])
    raise RuntimeError(f"Candidate did not define required function(s): {missing}")

print("Candidate loaded.")


In [ ]:
# ==============================================================================
# OPTIONAL OFFLINE TEST: Load candidate_code.py from local folder (no Drive)
# ==============================================================================

import os
from pathlib import Path

def _find_candidate_code() -> Path:
    candidates = [
        Path.cwd() / "candidate_code.py",
        Path.cwd() / "./candidate_code.py",
        Path("/content") / "candidate_code.py",   # Colab default dir
    ]
    for p in candidates:
        if p.exists() and p.is_file():
            return p
    raise FileNotFoundError(
        "candidate_code.py not found.\n"
        "Expected it in the same folder as Bench.ipynb (current working directory),\n"
        "or at /content/candidate_code.py (Colab).\n"
        "If you’re running locally, ensure candidate_code.py is in the unzipped folder."
    )

candidate_path = _find_candidate_code()
print("Loading candidate from:", candidate_path.resolve())

with open(candidate_path, "r", encoding="utf-8") as f:
    candidate_code = f.read()

exec(compile(candidate_code, str(candidate_path), "exec"), globals(), globals())

required = ["equilibrium_temperature", "simulate_temperature", "calibrate_T0_k"]
missing = [fn for fn in required if fn not in globals() or not callable(globals().get(fn))]
if missing:
    raise RuntimeError(f"Candidate did not define required function(s): {missing}")

print("Candidate loaded.")

In [ ]:

# ==============================================================================
# SUBTASK B
# ==============================================================================

def _ref_alpha_of_T(T: float, alpha_warm: float, alpha_cold: float, T0: float, k: float) -> float:
    if k <= 0:
        raise ValueError("k must be > 0")
    x = (T - T0) / k
    if x > 50:
        frac = 0.0
    elif x < -50:
        frac = 1.0
    else:
        frac = 1.0 / (1.0 + math.exp(x))
    alpha = alpha_warm + (alpha_cold - alpha_warm) * frac
    return max(0.0, min(1.0, alpha))


def _ref_energy_balance(T: float, S: float, sigma: float, alpha_warm: float, alpha_cold: float, T0: float, k: float) -> float:
    alpha = _ref_alpha_of_T(T, alpha_warm, alpha_cold, T0, k)
    return (1.0 - alpha) * (S / 4.0) - sigma * (T ** 4)


def _ref_equilibrium_temperature(S, sigma, alpha_warm, alpha_cold, T0, k, bracket, tol, max_iter):
    a, b = float(bracket[0]), float(bracket[1])
    fa = _ref_energy_balance(a, S, sigma, alpha_warm, alpha_cold, T0, k)
    fb = _ref_energy_balance(b, S, sigma, alpha_warm, alpha_cold, T0, k)
    if fa == 0.0: return a
    if fb == 0.0: return b
    if fa * fb > 0: raise ValueError("No sign change")
    lo, hi = a, b
    flo = fa
    for _ in range(int(max_iter)):
        mid = 0.5*(lo+hi)
        fmid = _ref_energy_balance(mid, S, sigma, alpha_warm, alpha_cold, T0, k)
        if abs(fmid) < tol or abs(hi-lo) < tol:
            return mid
        if flo * fmid <= 0:
            hi = mid
        else:
            lo = mid
            flo = fmid
    return 0.5*(lo+hi)


def _ref_simulate_temperature_series(C, S0, a, P_days, sigma, alpha_warm, alpha_cold, T0, k,
                                    T_init=288.0, dt_days=0.25, n_days=365*6):
    dt_sec = float(dt_days) * 86400.0
    P_sec = float(P_days) * 86400.0
    n_steps = int(round(float(n_days) / float(dt_days)))

    def S_of_t(t_sec: float) -> float:
        return float(S0) * (1.0 + float(a) * math.cos(2.0 * math.pi * (t_sec / P_sec)))

    def dTdt(t_sec: float, T: float) -> float:
        S_t = S_of_t(t_sec)
        alpha = _ref_alpha_of_T(T, alpha_warm, alpha_cold, T0, k)
        incoming = (1.0 - alpha) * (S_t / 4.0)
        outgoing = sigma * (T ** 4)
        return (incoming - outgoing) / float(C)

    t_days = np.zeros(n_steps + 1, dtype=float)
    T = np.zeros(n_steps + 1, dtype=float)
    T[0] = float(T_init)

    for i in range(n_steps):
        t0 = i * dt_sec
        Ti = T[i]
        # RK4
        k1 = dTdt(t0, Ti)
        k2 = dTdt(t0 + 0.5*dt_sec, Ti + 0.5*dt_sec*k1)
        k3 = dTdt(t0 + 0.5*dt_sec, Ti + 0.5*dt_sec*k2)
        k4 = dTdt(t0 + dt_sec, Ti + dt_sec*k3)
        T_next = Ti + (dt_sec/6.0)*(k1 + 2*k2 + 2*k3 + k4)
        if not math.isfinite(T_next) or T_next <= 0:
            raise ValueError("Unphysical temperature")
        T[i+1] = T_next
        t_days[i+1] = (i+1) * float(dt_days)
    return t_days, T


def _ref_simulate_temperature(C, S0, a, P_days, sigma, alpha_warm, alpha_cold, T0, k,
                              T_init=288.0, dt_days=0.25, n_days=365*6):
    t_days, T = _ref_simulate_temperature_series(
        C=C, S0=S0, a=a, P_days=P_days, sigma=sigma,
        alpha_warm=alpha_warm, alpha_cold=alpha_cold, T0=T0, k=k,
        T_init=T_init, dt_days=dt_days, n_days=n_days
    )
    final_start = float(n_days) - 365.0
    T_final = T[t_days >= final_start]
    return {"T_mean": float(np.mean(T_final)), "T_amp": float(0.5*(np.max(T_final)-np.min(T_final)))}


def _ref_calibrate_T0_k(observations, times_days, C, S0, a, P_days, sigma, alpha_warm, alpha_cold,
                        T_init=288.0, dt_days=0.25, search_bounds=((250.0, 290.0), (2.0, 20.0))):
    obs = np.asarray(observations, dtype=float)
    t_obs = np.asarray(times_days, dtype=float)
    (T0_lo, T0_hi), (k_lo, k_hi) = search_bounds

    dt_cal = 0.5
    dt_sec = dt_cal * 86400.0
    P_sec = float(P_days) * 86400.0
    n_days_needed = float(np.max(t_obs) + 10.0)
    n_steps = int(math.ceil(n_days_needed / dt_cal))
    t_series = np.arange(n_steps + 1, dtype=float) * dt_cal

    def S_of_t_sec(t_sec: float) -> float:
        return float(S0) * (1.0 + float(a) * math.cos(2.0 * math.pi * (t_sec / P_sec)))

    def simulate_candidate(T0, k):
        T = float(T_init)
        T_out = np.zeros(n_steps + 1, dtype=float)
        T_out[0] = T
        for i in range(n_steps):
            t_sec = (i * dt_cal) * 86400.0
            S_t = S_of_t_sec(t_sec)
            alpha = _ref_alpha_of_T(T, alpha_warm, alpha_cold, float(T0), float(k))
            incoming = (1.0 - alpha) * (S_t / 4.0)
            outgoing = sigma * (T ** 4)
            dTdt = (incoming - outgoing) / float(C)
            T = T + dTdt * dt_sec
            if not math.isfinite(T) or T <= 0:
                return None
            T_out[i+1] = T
        return T_out

    def sse_for(T0, k):
        T_out = simulate_candidate(T0, k)
        if T_out is None:
            return float("inf")
        pred = np.interp(t_obs, t_series, T_out)
        err = pred - obs
        return float(np.sum(err*err))

    T0_grid_1 = np.linspace(T0_lo, T0_hi, 13)
    k_grid_1  = np.linspace(k_lo,  k_hi,  13)
    best = (None, None, float("inf"))
    for T0 in T0_grid_1:
        for k in k_grid_1:
            v = sse_for(T0, k)
            if v < best[2]:
                best = (float(T0), float(k), float(v))

    T0_c, k_c, _ = best
    T0_grid_2 = np.linspace(max(T0_lo, T0_c-2.0), min(T0_hi, T0_c+2.0), 21)
    k_grid_2  = np.linspace(max(k_lo,  k_c-2.0),  min(k_hi,  k_c+2.0),  21)
    best2 = (T0_c, k_c, best[2])
    for T0 in T0_grid_2:
        for k in k_grid_2:
            v = sse_for(T0, k)
            if v < best2[2]:
                best2 = (float(T0), float(k), float(v))

    return {"T0_hat": best2[0], "k_hat": best2[1], "sse": best2[2]}


# Synthetic observations (hi-fi model)
rng = np.random.default_rng(OBS_SEED)
t_dense, T_dense = _ref_simulate_temperature_series(
    C=C, S0=S0, a=A_SEASON, P_days=P_DAYS, sigma=SIGMA,
    alpha_warm=ALPHA_WARM, alpha_cold=ALPHA_COLD,
    T0=T0_TRUE, k=K_TRUE,
    T_init=T_INIT, dt_days=DT_DAYS, n_days=N_DAYS
)
T_clean = np.interp(times_days, t_dense, T_dense)
observations = T_clean + rng.normal(0.0, NOISE_STD, size=T_clean.shape)

T_star_ref = _ref_equilibrium_temperature(S_A, SIGMA, ALPHA_WARM, ALPHA_COLD, T0_TRUE, K_TRUE, BRACKET_A, TOL_A, MAX_ITER_A)
metrics_ref = _ref_simulate_temperature(C, S0, A_SEASON, P_DAYS, SIGMA, ALPHA_WARM, ALPHA_COLD, T0_TRUE, K_TRUE, T_INIT, DT_DAYS, N_DAYS)
calib_ref = _ref_calibrate_T0_k(observations, times_days, C, S0, A_SEASON, P_DAYS, SIGMA, ALPHA_WARM, ALPHA_COLD, T_INIT, DT_DAYS, (T0_BOUNDS, K_BOUNDS))

print("Ref A:", T_star_ref)
print("Ref B:", metrics_ref)
print("Ref C:", calib_ref)


In [ ]:
# ==============================================================================
# SUBTASK C
# ==============================================================================
for fn in ["equilibrium_temperature","simulate_temperature","calibrate_T0_k"]:
    if fn not in globals():
        raise RuntimeError(f"Missing {fn}. Ensure the Candidate cell ran and printed 'Candidate loaded.'")


import unittest

class TestMiniBenchCandidate(unittest.TestCase):
    def test_A(self):
        T = float(equilibrium_temperature(S_A, SIGMA, ALPHA_WARM, ALPHA_COLD, T0_TRUE, K_TRUE, BRACKET_A, TOL_A, MAX_ITER_A))
        resid = _ref_energy_balance(T, S_A, SIGMA, ALPHA_WARM, ALPHA_COLD, T0_TRUE, K_TRUE)
        self.assertLess(abs(resid), 1e-4)
        self.assertLess(abs(T - T_star_ref), 0.05)

    def test_A_bad_bracket(self):
        with self.assertRaises(Exception):
            equilibrium_temperature(S_A, SIGMA, ALPHA_WARM, ALPHA_COLD, T0_TRUE, K_TRUE, (330.0, 340.0), TOL_A, MAX_ITER_A)

    def test_B(self):
        out = simulate_temperature(C, S0, A_SEASON, P_DAYS, SIGMA, ALPHA_WARM, ALPHA_COLD, T0_TRUE, K_TRUE, T_INIT, DT_DAYS, N_DAYS)
        self.assertLess(abs(float(out["T_mean"]) - metrics_ref["T_mean"]), 0.2)
        self.assertLess(abs(float(out["T_amp"])  - metrics_ref["T_amp"]), 0.2)

    def test_B_determinism(self):
        o1 = simulate_temperature(C, S0, A_SEASON, P_DAYS, SIGMA, ALPHA_WARM, ALPHA_COLD, T0_TRUE, K_TRUE, T_INIT, DT_DAYS, N_DAYS)
        o2 = simulate_temperature(C, S0, A_SEASON, P_DAYS, SIGMA, ALPHA_WARM, ALPHA_COLD, T0_TRUE, K_TRUE, T_INIT, DT_DAYS, N_DAYS)
        self.assertEqual(o1, o2)

    def test_C(self):
        out = calibrate_T0_k(observations, times_days, C, S0, A_SEASON, P_DAYS, SIGMA, ALPHA_WARM, ALPHA_COLD, T_INIT, DT_DAYS, (T0_BOUNDS, K_BOUNDS))
        self.assertLess(abs(float(out["T0_hat"]) - calib_ref["T0_hat"]), 2.0)
        self.assertLess(abs(float(out["k_hat"])  - calib_ref["k_hat"]), 1.5)
        self.assertLessEqual(float(out["sse"]), 1.2 * float(calib_ref["sse"]))

    def test_C_determinism(self):
        o1 = calibrate_T0_k(observations, times_days, C, S0, A_SEASON, P_DAYS, SIGMA, ALPHA_WARM, ALPHA_COLD, T_INIT, DT_DAYS, (T0_BOUNDS, K_BOUNDS))
        o2 = calibrate_T0_k(observations, times_days, C, S0, A_SEASON, P_DAYS, SIGMA, ALPHA_WARM, ALPHA_COLD, T_INIT, DT_DAYS, (T0_BOUNDS, K_BOUNDS))
        self.assertEqual(o1, o2)

unittest.main(argv=["-v"], exit=False)


RuntimeError: Missing equilibrium_temperature. Ensure the Candidate cell ran and printed 'Candidate loaded.'